In [1]:
"""
Step 2: controlled experiments on how to merge CICIDS2018 into CICIDS2017.

Same model family as newmain.ipynb (RF + XGBoost soft vote, RobustScaler,
balanced sample weights) but smaller so each run takes minutes.

Experiments (each retrains from scratch, scaler fitted on its own train set):
  E0  2017 only                              (baseline, reproduces the gap)
  E1  2017 + 2018 attacks only               (your original idea)
  E4  2017 + duplicated 2017 attacks         (same added row count as E1, but no new
                                              diversity: separates "more records" from
                                              "records from a different network")
  E2  2017 + 2018 attacks + 2018 benign      (benign matched to attack count)
  E3  E2 without the K most shifted features (tests the "domain shift" idea)

Honest test set: the LAST part of every (Source File, Raw Label) block in 2018
is held out, with a small gap before it, so neighbouring near-duplicate flows
do not leak into training. 2017 gets a random 10% test split (in-distribution
check only).
"""
import warnings
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')


In [2]:

# ---------------- CONFIG ----------------
PATH_2017 = r'Preprocessing/cicids2017_preprocessed.csv'
PATH_2018 = r'Preprocessing/cicids2018_extra.csv'
CAP_2017 = 60000        # max rows per class taken from 2017 (speed)
CAP_2018 = 20000        # max rows per 2018 attack class used for training
BENIGN_RATIO = 1.0      # 2018 benign rows = ratio x 2018 attack rows (E2/E3)
TEST_FRAC = 0.30        # last 30% of each 2018 block is test
GAP_FRAC = 0.02         # 2% of each block discarded between train and test
DROP_K = 5              # E3: remove this many most shifted features
SEED = 42


In [3]:
# ----------------------------------------

CLASSES = ['Brute Force', 'DDoS', 'DoS', 'Normal Traffic', 'Port Scan']
META = ['Attack Type', 'Raw Label', 'Source File']

d18 = pd.read_csv(PATH_2018)
FEATURES = [c for c in d18.columns if c not in META]
d17 = pd.read_csv(PATH_2017, usecols=FEATURES + ['Attack Type'])
print(f'2017: {d17.shape}   2018: {d18.shape}   features: {len(FEATURES)}')


def cap_per_class(df, n):
    out = [g.sample(min(len(g), n), random_state=SEED) for _, g in df.groupby('Attack Type')]
    return pd.concat(out)


def block_split(df):
    tr, te = [], []
    for _, g in df.groupby(['Source File', 'Raw Label'], sort=False):
        n = len(g)
        cut = int(n * (1 - TEST_FRAC))
        gap = int(n * GAP_FRAC)
        tr.append(g.iloc[:cut])
        te.append(g.iloc[cut + gap:])
    return pd.concat(tr), pd.concat(te)



2017: (2515963, 41)   2018: (3999643, 43)   features: 40


In [4]:

# 2017: capped sample, random 90/10 split
s17 = cap_per_class(d17, CAP_2017)
tr17, te17 = train_test_split(s17, test_size=0.10, stratify=s17['Attack Type'], random_state=SEED)

# 2018: block split, then separate attack and benign training rows
tr18, te18 = block_split(d18)
tr18_att = cap_per_class(tr18[tr18['Attack Type'] != 'Normal Traffic'], CAP_2018)
tr18_ben_all = tr18[tr18['Attack Type'] == 'Normal Traffic']

print('\n2018 train attack rows used:\n', tr18_att['Attack Type'].value_counts().to_string())
print('\n2018 TEST rows:\n', te18['Attack Type'].value_counts().to_string())

# ---- feature shift: 2017 benign vs 2018 benign ----
b17 = tr17[tr17['Attack Type'] == 'Normal Traffic'].sample(20000, random_state=SEED, replace=False) \
    if (tr17['Attack Type'] == 'Normal Traffic').sum() >= 20000 else tr17[tr17['Attack Type'] == 'Normal Traffic']
b18 = tr18_ben_all.sample(min(20000, len(tr18_ben_all)), random_state=SEED)
ks = pd.Series({f: ks_2samp(b17[f], b18[f]).statistic for f in FEATURES}).sort_values(ascending=False)
print('\nMost shifted features (KS statistic, 2017 benign vs 2018 benign; 1.0 = totally different):')
print(ks.head(12).round(3).to_string())
shifted = ks.head(DROP_K).index.tolist()



2018 train attack rows used:
 Attack Type
Brute Force    20000
DDoS           20000
DoS            20000

2018 TEST rows:
 Attack Type
Normal Traffic    982372
DDoS               56167
DoS                55025
Brute Force        26348

Most shifted features (KS statistic, 2017 benign vs 2018 benign; 1.0 = totally different):
Fwd IAT Min              0.401
Fwd Packet Length Min    0.386
Bwd IAT Min              0.381
Min Packet Length        0.379
Bwd Packet Length Std    0.368
Fwd Packet Length Std    0.363
min_seg_size_forward     0.355
Bwd IAT Max              0.352
Bwd IAT Total            0.351
Fwd Packet Length Max    0.349
Fwd IAT Std              0.342
Bwd Packet Length Max    0.339


In [5]:
def train_and_eval(train, feats, name):
    le = LabelEncoder().fit(CLASSES)
    y = le.transform(train['Attack Type'])
    sc = RobustScaler().fit(train[feats].values)
    X = sc.transform(train[feats].values)
    w = compute_sample_weight('balanced', y)

    rf = RandomForestClassifier(n_estimators=100, max_depth=20, max_features=0.4,
                                n_jobs=-1, random_state=SEED).fit(X, y, sample_weight=w)
    xg = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, subsample=0.8,
                       colsample_bytree=0.5, tree_method='hist', random_state=SEED,
                       n_jobs=-1).fit(X, y, sample_weight=w)

    def predict(df):
        Xs = sc.transform(df[feats].values)
        P = (rf.predict_proba(Xs) + xg.predict_proba(Xs)) / 2
        return le.classes_[P.argmax(axis=1)]

    row = {'experiment': name, 'train_rows': len(train), 'n_features': len(feats)}

    p17 = predict(te17)
    row['2017 macroF1'] = f1_score(te17['Attack Type'], p17, average='macro')

    p18 = predict(te18)
    t18 = te18['Attack Type'].values
    present = [c for c in CLASSES if c in set(t18)]
    for c in present:
        row[f'2018 recall {c}'] = recall_score(t18 == c, p18 == c)
    row['2018 macroF1'] = f1_score(t18, p18, labels=present, average='macro')
    return row


def build_train(use_att, use_ben):
    parts = [tr17]
    if use_att:
        parts.append(tr18_att)
    if use_ben:
        n = int(BENIGN_RATIO * len(tr18_att))
        parts.append(tr18_ben_all.sample(min(n, len(tr18_ben_all)), random_state=SEED))
    return pd.concat(parts)


def build_dup():
    counts = tr18_att['Attack Type'].value_counts()
    att17 = tr17[tr17['Attack Type'].isin(counts.index)]
    extra = [g.sample(counts[c], replace=True, random_state=SEED)
             for c, g in att17.groupby('Attack Type')]
    return pd.concat([tr17] + extra)


rows = []
rows.append(train_and_eval(build_train(False, False), FEATURES, 'E0 2017 only'))
rows.append(train_and_eval(build_dup(), FEATURES, 'E4 +duplicated 2017 attacks'))
rows.append(train_and_eval(build_train(True, False), FEATURES, 'E1 +2018 attacks'))
rows.append(train_and_eval(build_train(True, True), FEATURES, 'E2 +2018 attacks+benign'))
keep = [f for f in FEATURES if f not in shifted]
rows.append(train_and_eval(build_train(True, True), keep, f'E3 E2 minus {DROP_K} shifted'))

res = pd.DataFrame(rows).set_index('experiment')
pd.set_option('display.width', 200)
print('\nRESULTS (2018 columns = held-out 2018 blocks; recall of Normal = 1 - false alarm rate)')
print(res.round(3).T.to_string())
print('\nShifted features removed in E3:', shifted)
res.to_csv('experiment_results.csv')
print('Saved experiment_results.csv')


RESULTS (2018 columns = held-out 2018 blocks; recall of Normal = 1 - false alarm rate)
experiment                  E0 2017 only  E4 +duplicated 2017 attacks  E1 +2018 attacks  E2 +2018 attacks+benign  E3 E2 minus 5 shifted
train_rows                    224235.000                   284235.000        284235.000               344235.000             344235.000
n_features                        40.000                       40.000            40.000                   40.000                 35.000
2017 macroF1                       1.000                        0.999             0.999                    0.999                  0.999
2018 recall Brute Force            0.000                        0.000             1.000                    1.000                  1.000
2018 recall DDoS                   0.000                        0.000             0.998                    0.988                  0.989
2018 recall DoS                    0.178                        0.123             1.000         